# 🏦 Who Will Invest on Monday, 20 July 2026 — and Roughly How Much?

**A next-day prediction on NSE / BSE bulk & block deals (2020 → 2026), presented honestly.**

Runs end-to-end on **Google Colab**. It will:
1. Load the deals data (upload it, mount Drive, or read a local path).
2. Label every *house* as a **market-maker** (buys ≈ sells, round-trips intraday) or a **net investor** (buys ≫ sells).
3. Engineer leakage-safe features and train on the **oldest 75%** of the timeline, test on the newest **25%**.
4. Compare the model against a dumb **'recently-active-houses-buy-again' baseline** — the honesty check.
5. Forecast **who buys on Mon 20-Jul-2026**, the tentative ₹-crore amount (a range, not a point), and render a **class dashboard**.

> *House* = the desk named in each large deal. *Invest* = a **BUY** (`is_purchase=1`).
> Read the caveats at the bottom before quoting any number.

## 1 · Setup

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib 2>/dev/null
import numpy as np, pandas as pd, sqlite3, os, json
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)
plt.rcParams['figure.dpi'] = 110
print('Ready.')

## 2 · Load the data

Give it **either** the CSV (`india_bulk_block_deals_2020_to_today.csv`) **or** the SQLite
(`india_large_deals.sqlite`). On Colab, the cell shows an upload button if the file isn't
already there; or set `DATA_DIR` to a Google-Drive folder.

In [ ]:
DATA_DIR    = ''  # e.g. '/content/drive/MyDrive/deals'  (leave '' for cwd / upload)
CSV_NAME    = 'india_bulk_block_deals_2020_to_today.csv'
SQLITE_NAME = 'india_large_deals.sqlite'

def _find(name):
    for base in ([DATA_DIR] if DATA_DIR else []) + ['.', '/content']:
        p = os.path.join(base, name)
        if os.path.exists(p): return p
    return None

csv_path, sql_path = _find(CSV_NAME), _find(SQLITE_NAME)
if not csv_path and not sql_path:
    try:
        from google.colab import files
        print('Upload the CSV or SQLite file...'); up = files.upload()
        for fn in up:
            if fn.endswith('.csv'): csv_path = fn
            elif fn.endswith(('.sqlite','.db')): sql_path = fn
    except Exception as e:
        raise SystemExit('No data file. Set DATA_DIR or upload it. ' + str(e))

COLS = ['deal_date','client_name','symbol','is_purchase','trade_value_crore']
if sql_path:
    con = sqlite3.connect(sql_path)
    allrows = pd.read_sql_query('SELECT %s FROM deals' % ','.join(COLS), con); con.close()
    print('Loaded from SQLite:', sql_path)
else:
    allrows = pd.read_csv(csv_path, encoding='utf-8-sig', usecols=COLS)
    print('Loaded from CSV:', csv_path)
allrows['deal_date'] = pd.to_datetime(allrows['deal_date'])
print(f'{len(allrows):,} rows | {allrows.deal_date.min().date()} -> {allrows.deal_date.max().date()} | '
      f'{allrows.client_name.nunique():,} houses')
allrows.head(3)

## 3 · Config + label each house: market-maker vs investor

A house that **sells about as much as it buys** is a *market-maker* — it round-trips within the
day and ends flat, so its 'buy' is **turnover, not investment**. A house that **buys far more
than it sells** is a *net investor* actually deploying capital. We use the whole buy/sell history
to tag each one — this is the difference the headline number must respect.

In [ ]:
ASOF, TARGET, TARGET_DOW = '2026-07-15', '2026-07-20', 0  # Monday
allrows = allrows[allrows.deal_date <= pd.Timestamp(ASOF)]
allrows = allrows[allrows.client_name.notna() & (allrows.client_name!='')]
buys = allrows[allrows.is_purchase==1].copy()

g = allrows.assign(bcr=np.where(allrows.is_purchase==1, allrows.trade_value_crore, 0.0),
                   scr=np.where(allrows.is_purchase==0, allrows.trade_value_crore, 0.0))
lt = g.groupby('client_name')[['bcr','scr']].sum().rename(columns={'bcr':'buy_cr','scr':'sell_cr'})
lt['sb'] = lt.sell_cr / lt.buy_cr.replace(0, np.nan)
def htype(r):
    if r.buy_cr < 50: return 'Small / occasional'
    if pd.isna(r.sb) or r.sb < 0.6: return 'Net investor (accumulator)'
    if r.sb > 1.6: return 'Net seller'
    return 'Market-maker (round-trips)'
lt['house_type'] = lt.apply(htype, axis=1)
print(lt.house_type.value_counts())

## 4 · Leakage-safe feature engineering

Keep only buys; build a per-house / per-day buy total as a matrix (rows = trading days,
cols = houses). Every feature at day *T* uses only **trailing** windows that end at *T* — never
the future. The amount estimate is a **trailing median** of a house's active-day buy sizes
(medians shrug off the occasional giant block that would wreck a mean).

In [ ]:
daily = (buys.groupby(['client_name','deal_date'], as_index=False).trade_value_crore.sum()
              .rename(columns={'trade_value_crore':'buy_cr'}))
cal = pd.Index(sorted(buys.deal_date.unique()), name='deal_date'); cal_list=list(cal); n_days=len(cal_list)
bd = daily.groupby('client_name').deal_date.nunique()
universe = sorted(bd[bd>=5].index)                    # >=5 buy-days ever (no survivorship filter)
piv = (daily[daily.client_name.isin(universe)]
       .pivot_table(index='deal_date', columns='client_name', values='buy_cr', aggfunc='sum')
       .reindex(cal).fillna(0.0))
active = (piv>0).astype(np.int8); masked = piv.where(active>0)
print(f'{n_days} trading days | universe {len(universe)} houses')

In [ ]:
WINDOWS=[5,10,20,60,120,252]
roll_rate={w: active.rolling(w,min_periods=1).mean() for w in WINDOWS}
amt_mean20=piv.rolling(20,min_periods=1).sum()/active.rolling(20,min_periods=1).sum().replace(0,np.nan)
amt_mean60=piv.rolling(60,min_periods=1).sum()/active.rolling(60,min_periods=1).sum().replace(0,np.nan)
amt_med90 =masked.rolling(90 ,min_periods=3).median()   # ROBUST amount
amt_med252=masked.rolling(252,min_periods=3).median()
amt_p25   =masked.rolling(90 ,min_periods=3).quantile(0.25)
amt_p75   =masked.rolling(90 ,min_periods=3).quantile(0.75)
day_index=pd.Series(np.arange(n_days),index=cal)
last_idx=active.mul(day_index.values,axis=0).where(active>0).cummax().ffill()
days_since=(day_index.values.reshape(-1,1)-last_idx.values)
dow=pd.Series([d.weekday() for d in cal_list],index=cal); is_mon=(dow==TARGET_DOW).values.reshape(-1,1)
mon_act=pd.DataFrame(active.values*is_mon,index=cal,columns=active.columns)
mon_pre=pd.DataFrame(np.repeat(is_mon,active.shape[1],axis=1),index=cal,columns=active.columns)
mon_rate=(mon_act.rolling(252,min_periods=1).sum()/mon_pre.rolling(252,min_periods=1).sum().replace(0,np.nan)).fillna(0.0)
def features_asof(i):
    f=pd.DataFrame(index=active.columns)
    for w in WINDOWS: f[f'rate_{w}']=roll_rate[w].iloc[i].values
    f['trend_20_60']=f['rate_20']-f['rate_60']; f['days_since']=days_since[i]
    f['recency']=np.exp(-f['days_since']/20.0)
    f['amt_mean20']=amt_mean20.iloc[i].values; f['amt_mean60']=amt_mean60.iloc[i].values
    f['mon_rate']=mon_rate.iloc[i].values; f['log_amt60']=np.log1p(f['amt_mean60'].fillna(0))
    return f
FEAT=[f'rate_{w}' for w in WINDOWS]+['trend_20_60','days_since','recency','amt_mean20','amt_mean60','mon_rate','log_amt60']
print('features:',FEAT)

## 5 · Build the panel, split 75 / 25, train — and check honesty vs a dumb baseline

Each row = *(house, day)* → **did this house buy the next trading day?** Oldest 75% of days
train, newest 25% test. Then the reality check: how much better is the model than simply
**ranking houses by how often they bought in the last 20 sessions**?

In [ ]:
rows=[]
for i in range(252,n_days-1):
    live=roll_rate[252].iloc[i].values>0
    f=features_asof(i)[live].copy(); f['label']=active.iloc[i+1].values[live]; f['asof_i']=i; rows.append(f)
panel=pd.concat(rows,ignore_index=True)
asof=sorted(panel.asof_i.unique()); split_i=asof[int(round(len(asof)*0.75))]
train=panel[panel.asof_i<split_i]; test=panel[panel.asof_i>=split_i].copy()
clf=HistGradientBoostingClassifier(max_depth=4,learning_rate=0.06,max_iter=400,
        l2_regularization=1.0,min_samples_leaf=40,random_state=0).fit(train[FEAT],train.label)
test['p']=clf.predict_proba(test[FEAT])[:,1]

def pk(df,score,k):
    per=[g.nlargest(k,score).label.mean() for _,g in df.groupby('asof_i')]
    return np.mean(per),np.median(per),np.percentile(per,10)
print(f"base rate (a house buying next day)      : {test.label.mean():.1%}")
print(f"MODEL     AUC {roc_auc_score(test.label,test.p):.3f} | Brier {brier_score_loss(test.label,test.p):.3f}")
print(f"BASELINE  AUC {roc_auc_score(test.label,test.rate_20):.3f}  (rank by last-20-day buy frequency)")
for k in [5,10,20]:
    mm,mmed,mp10=pk(test,'p',k); bm,_,_=pk(test,'rate_20',k)
    print(f"Precision@{k:<2}: model {mm:.2f} (median {mmed:.2f}, worst-10% {mp10:.2f})  vs baseline {bm:.2f}  -> lift {mm-bm:+.3f}")
novel=test[(test.rate_5==0)&(test.label==1)]
print(f"\n{len(novel)/max(1,test.label.sum()):.0%} of next-day buyers were INACTIVE the prior week ('new' buyers).")
print('The model is built to spot regulars, not surprises — keep that in mind.')

## 6 · Does the model tell the truth? (calibration + what drives it)

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
tb=test.copy(); tb['b']=pd.qcut(tb.p,10,duplicates='drop')
c=tb.groupby('b',observed=True).agg(pred=('p','mean'),actual=('label','mean'))
mx=c.pred.max(); ax[0].plot([0,mx],[0,mx],'--',c='gray',label='perfect')
ax[0].plot(c.pred,c.actual,'o-',c='#0e7c66'); ax[0].set_xlabel('Predicted probability')
ax[0].set_ylabel('Actual buy rate'); ax[0].set_title('Calibration — is a 30% call really ~30%?'); ax[0].legend()
corr=pd.Series({col:abs(np.corrcoef(train[col].fillna(0),train.label)[0,1]) for col in FEAT}).sort_values()
ax[1].barh(corr.index,corr.values,color='#1e5f8c'); ax[1].set_title('What signals matter most'); 
plt.tight_layout(); plt.show()

## 7 · 🔮 The forecast for Monday 20 July 2026

Retrain on the whole timeline, score every live house as-of 15-Jul. We show the amount as a
**typical-day median with a p25–p75 range**, and only attach a rupee figure when a buy is at
least plausible (probability ≥ 30%). Colour = what kind of house it is.

In [ ]:
clf_f=HistGradientBoostingClassifier(max_depth=4,learning_rate=0.06,max_iter=400,
        l2_regularization=1.0,min_samples_leaf=40,random_state=0).fit(panel[FEAT],panel.label)
i=n_days-1; live=roll_rate[252].iloc[i].values>0; ff=features_asof(i)[live].copy(); ff['house']=ff.index
ff['p_buy']=clf_f.predict_proba(ff[FEAT])[:,1]
ff['amt']=amt_med90.iloc[i].reindex(ff.index).fillna(amt_med252.iloc[i].reindex(ff.index)).fillna(amt_mean60.iloc[i].reindex(ff.index)).fillna(0)
ff['lo']=amt_p25.iloc[i].reindex(ff.index); ff['hi']=amt_p75.iloc[i].reindex(ff.index)
ff['exp_flow']=np.where(ff.p_buy>=0.30, ff.p_buy*ff.amt, np.nan)
ff['type']=ff.house.map(lt.house_type).fillna('Small / occasional')
win=cal_list[max(0,i-20)]
fav=(buys[buys.deal_date>=win].groupby(['client_name','symbol']).trade_value_crore.sum().reset_index()
     .sort_values('trade_value_crore',ascending=False).drop_duplicates('client_name').set_index('client_name').symbol)
ff['likely_stock']=ff.house.map(fav).fillna('-')
likely=ff.sort_values('p_buy',ascending=False).head(12)
show=likely[['house','p_buy','amt','lo','hi','type','likely_stock']].rename(columns={
     'house':'House','p_buy':'Buy prob.','amt':'Typical buy Rs cr','lo':'p25','hi':'p75',
     'type':'What kind of house','likely_stock':'Likely stock'})
show.index=range(1,len(show)+1)
show.style.format({'Buy prob.':'{:.0%}','Typical buy Rs cr':'{:,.0f}','p25':'{:,.0f}','p75':'{:,.0f}'})\
    .background_gradient(subset=['Buy prob.'],cmap='Greens')\
    .set_caption('MOST LIKELY large-deal BUYERS on Monday 20-Jul-2026')

## 8 · Dashboard view

In [ ]:
cmap={'Market-maker (round-trips)':'#1e5f8c','Net investor (accumulator)':'#0e7c66',
      'Net seller':'#b45309','Small / occasional':'#94a3b8'}
fig,ax=plt.subplots(1,2,figsize=(13,5))
t=likely.head(8)[::-1]
ax[0].barh(t.house.str.slice(0,24), t.amt, xerr=[t.amt-t.lo, t.hi-t.amt],
           color=[cmap.get(x,'#94a3b8') for x in t.type], capsize=3)
ax[0].set_title('Typical BUY size on a buy-day (Rs cr, p25-p75)'); ax[0].set_xlabel('Rs crore')
sc=ff[ff.p_buy>=0.05]
ax[1].scatter(sc.p_buy, sc.amt.clip(1), s=30, alpha=.6, color=[cmap.get(x,'#94a3b8') for x in sc.type])
for _,r in likely.head(6).iterrows(): ax[1].annotate(r.house.split()[0],(r.p_buy,max(r.amt,1)),fontsize=8)
ax[1].set_xlabel('Probability it trades on 20-Jul'); ax[1].set_ylabel('Typical buy size (Rs cr)')
ax[1].set_yscale('log'); ax[1].set_title('Sure-and-steady desks  vs  rare-but-huge funds')
plt.tight_layout(); plt.show()
gross=ff[(ff.p_buy>=0.5)].exp_flow.sum(); sure=(ff.p_buy>=0.8).sum()
inv=(likely.type=='Net investor (accumulator)').sum()
print(f'Near-certain buyers (>=80% prob): {sure} houses  — and {inv}/12 of the top list are real investors.')
print(f'Expected GROSS large-deal BUY turnover from likely buyers (>=50% prob): ~Rs {gross:,.0f} crore')
print('NOTE: "turnover", not "investment" — the top names sell about as much as they buy the same day.')

## 9 · How to read this — and the honest caveats

**The answer.** On Monday 20-Jul-2026 the houses almost certain to appear as large-deal buyers
are a handful of high-frequency desks — **Junomoneta, HRTI, QE Securities, Microcurves, NK
Securities, Jump Trading, Graviton** — each buying roughly **₹90–160 cr** on a typical day.

**But read the fine print:**
- **These are market-makers, not investors.** Over their whole history they *sell about as much
  as they buy* (sell/buy ≈ 1.0), so they end the day roughly flat. Their 'buy' is trading
  turnover, not committed capital. Every name in the top list is tagged this way — and **0 of
  the top 12 are net investors.**
- **Real investors (ICICI Pru, SBI, HDFC mutual funds) buy in huge size but rarely** and on
  unpredictable days, so they score a low probability. The model can't call *their* Monday.
- **The model mostly re-identifies regulars.** It beats a dumb 'recently-active' baseline only
  modestly (AUC ≈ 0.89 vs 0.86). About a third of each day's buyers are 'new' (inactive the prior
  week) and the model catches almost none of them. It predicts **habits, not surprises.**
- **Amounts are typical-day medians with wide ranges** (~±50%). Treat them as ballparks.
- Assumes 20-Jul is a normal session (no surprise holiday or market-moving event).

*Built from public NSE/BSE bulk & block deal disclosures. Educational use only — not trading advice.*